# 01 · 서버 환경 설치 및 KiTS 2023 데이터 준비
기존 Python 3.11 venv를 이 노트북의 커널로 선택하세요. 새 가상환경은 생성하지 않습니다.
Linux NVIDIA 서버 전용이며 개발용 Mac에서는 설치 전에 중단됩니다.
순서: 기존 venv에 의존성 설치 → GPU 확인·서버 설정 생성 → 공식 학습용 489건 다운로드 → 변환·전처리 → 5-fold 고정.
데이터는 CC BY-NC-SA 조건이며 [공식 안내](https://github.com/neheller/kits23#license-and-attribution)를 따릅니다.
코드/데이터 버전을 기록하지만 논문 당시 환경과의 완전한 동일성은 아직 확인되지 않았습니다.

## 0-1. 기존 venv와 서버 확인
`STORAGE_ROOT`를 서버 저장 경로로 수정하세요. 데이터와 설치 캐시는 Git 저장소 밖에 둡니다.
Python 3.11 venv, NVIDIA 드라이버(CUDA 12.8 지원), git, tmux가 필요합니다.
이 셀은 파일을 생성하지 않고 환경만 검사합니다. 기존 config.server.json이 있으면 그 저장 경로를 사용하세요.

In [ ]:
from pathlib import Path
import sys, os, json, shutil, subprocess
sys.dont_write_bytecode = True
STORAGE_ROOT = Path("/root/nnunet-data")  # 서버에 맞게 수정

if sys.platform != "linux" or shutil.which("nvidia-smi") is None:
    raise RuntimeError("Linux NVIDIA GPU 서버에서만 실행하세요. 설치하지 않았습니다.")
if sys.prefix == sys.base_prefix:
    raise RuntimeError("기존 venv의 Python을 노트북 커널로 선택하세요.")
if sys.version_info[:2] != (3, 11):
    raise RuntimeError("이 재현 환경은 Python 3.11 venv를 사용합니다.")
for command in ("git", "tmux"):
    if shutil.which(command) is None:
        raise RuntimeError(f"서버에 {command}를 먼저 설치하세요.")
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "nnunetv2").is_dir())
EXPERIMENT = REPO / "experiments/kits2023"
if not STORAGE_ROOT.is_absolute():
    raise ValueError("STORAGE_ROOT는 절대 경로여야 합니다.")
STORAGE_ROOT = STORAGE_ROOT.resolve()
if STORAGE_ROOT == REPO or REPO in STORAGE_ROOT.parents:
    raise ValueError("저장 경로는 Git 저장소 밖이어야 합니다.")
server_config = EXPERIMENT / "config.server.json"
if server_config.exists():
    existing = json.loads(server_config.read_text())
    if Path(existing["storage_root"]).resolve() != STORAGE_ROOT:
        raise ValueError("기존 config.server.json과 STORAGE_ROOT가 다릅니다. 경로를 맞추세요.")
print("Kernel Python:", sys.executable)
print("Storage:", STORAGE_ROOT)
print(subprocess.check_output(["nvidia-smi"], text=True))


## 0-2. 현재 커널의 venv에 패키지 설치
PyTorch 2.8.0 CUDA 12.8, 현재 checkout의 nnU-Net(editable), Jupyter·평가 패키지를 설치합니다.
현재 venv의 패키지가 변경됩니다. 설치용 캐시는 STORAGE_ROOT/cache에 저장합니다.
이미 torch/numpy 등을 불러온 커널이라면 먼저 커널을 재시작하고 0-1부터 실행하세요.
이 셀은 데이터 다운로드나 학습을 시작하지 않습니다.

In [ ]:
# 반드시 직전 서버 확인 셀을 먼저 실행하세요.
if sys.platform != "linux" or sys.prefix == sys.base_prefix:
    raise RuntimeError("Linux 서버의 기존 venv가 필요합니다.")
if any(name in sys.modules for name in ("torch", "numpy", "nnunetv2")):
    raise RuntimeError("설치 전 커널을 재시작하고 0-1부터 실행하세요.")
STORAGE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["PIP_CACHE_DIR"] = str(STORAGE_ROOT / "cache/pip")
os.environ["XDG_CACHE_HOME"] = str(STORAGE_ROOT / "cache")
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"

def pip_install(*args):
    subprocess.run([sys.executable, "-m", "pip", *args], check=True)

pip_install("install", "--upgrade", "pip")
pip_install("install", "torch==2.8.0", "--index-url", "https://download.pytorch.org/whl/cu128")
pip_install("install", "torch==2.8.0", "-e", str(REPO),
            "-r", str(EXPERIMENT / "requirements-server.txt"))
pip_install("check")


## 0-3. GPU 검증 및 서버 설정 생성
새 Python 프로세스에서 CUDA 인식을 확인합니다. 성공하면 실행을 허용하는 서버 설정을 생성합니다.
기존 config.server.json은 보존합니다. 설치 패키지 목록은 environment.lock.txt에 기록합니다.
다음 셀부터 기존 데이터 준비 흐름으로 이어집니다.

In [ ]:
subprocess.run([sys.executable, "-c",
    "import torch; "
    "assert torch.cuda.is_available(), 'CUDA unavailable: check NVIDIA driver and CUDA wheel'; "
    "print('GPU:', torch.cuda.get_device_name(0))"], check=True)
server_config = EXPERIMENT / "config.server.json"
if not server_config.exists():
    server_values = json.loads((EXPERIMENT / "config.json").read_text())
    server_values.update(
        storage_root=str(STORAGE_ROOT),
        source_dataset=str(STORAGE_ROOT / "source/kits23/dataset"),
        execution_enabled=True,
        expected_git_commit=subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip(),
    )
    server_config.write_text(json.dumps(server_values, indent=2) + "\n")
(STORAGE_ROOT / "environment.lock.txt").write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))
print("Server config:", server_config)
print("Environment lock:", STORAGE_ROOT / "environment.lock.txt")


## 1. 실험 설정 불러오기
**0-3까지 최초 설치를 마쳤으면 Kernel → Restart Kernel을 실행한 뒤 이 셀부터 계속하세요.**
editable 설치 경로(.pth)는 Python 시작 시 적용되므로 설치 당시 실행 중이던 커널에는 아직 반영되지 않을 수 있습니다.
재시작 후 0번 설치 단계를 반복할 필요는 없습니다. 아래 코드 셀은 필요한 변수와 설정을 다시 불러옵니다.


In [ ]:
from pathlib import Path
import sys, os, json, shlex, subprocess
sys.dont_write_bytecode = True
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "nnunetv2").is_dir())
EXPERIMENT = REPO / "experiments/kits2023"
sys.path.insert(0, str(EXPERIMENT / "scripts"))
from workflow import read_config, paths, guard, environment, model_dir
CONFIG = EXPERIMENT / "config.server.json"
if not CONFIG.exists():
    CONFIG = EXPERIMENT / "config.json"  # 로컬 기본 설정: 실행 비활성화
cfg = read_config(CONFIG)
print("Config:", CONFIG)
print("Execution enabled:", cfg["execution_enabled"])
print("Storage:", cfg["storage_root"])


## 서버 확인
GPU, RAM, 저장 공간을 확인합니다. 전처리 worker 수는 config.server.json에서 조절합니다.

In [ ]:
guard(cfg)
import shutil
print(subprocess.check_output(["nvidia-smi"], text=True))
print("Disk:", shutil.disk_usage(Path(cfg["storage_root"])))
print(subprocess.check_output(["free", "-h"], text=True))


## 다운로드부터 전처리까지 시작
아래 셀은 `tmux`에서 준비 작업을 시작합니다. `tmux`가 없으면 서버에서 설치하세요.
브라우저/커널 종료와 무관하게 작업이 유지됩니다. 동일 세션이 있으면 다시 시작하지 않습니다.
중단 후 같은 셀을 다시 실행하면 검증된 다운로드/완료된 전처리는 재사용합니다.

In [ ]:
guard(cfg)
from workflow import paths
p = paths(cfg)
session = "kits23-prepare"
log = p["root"] / "prepare.console.log"
if subprocess.run(["tmux", "has-session", "-t", session], capture_output=True).returncode == 0:
    print("Already running:", session)
else:
    command = [sys.executable, "-u", str(EXPERIMENT/"scripts/prepare_data.py"), "--config", str(CONFIG)]
    shell_command = shlex.join(command) + " >> " + shlex.quote(str(log)) + " 2>&1"
    subprocess.run(["tmux", "new-session", "-d", "-s", session, shell_command], check=True, env=environment(cfg))
    print("Started. Log:", log)


## 준비 상태 확인
이 셀을 다시 실행해 로그를 갱신합니다. `PREPARATION COMPLETE` 또는 완료 marker를 확인한 후 2번 노트북으로 이동합니다.

In [ ]:
p = paths(cfg)
log = p["root"] / "prepare.console.log"
if log.exists():
    from collections import deque
    with log.open(errors="replace") as f:
        print("".join(deque(f, maxlen=25)))
marker = p["pre"] / cfg["dataset_name"] / "preparation.complete.json"
print("Preparation complete:", marker.exists())
if marker.exists():
    plans = json.loads((marker.parent / "nnUNetPlans.json").read_text())
    conf = plans["configurations"][cfg["configuration"]]
    print({k: conf.get(k) for k in ["patch_size", "batch_size", "spacing"]})
    splits = json.loads((marker.parent / "splits_final.json").read_text())
    print([(i, len(s["train"]), len(s["val"])) for i, s in enumerate(splits)])
